# 02 · Fault injection — EMFI (electromagnetic)

**Electromagnetic fault injection** upsets a chip at a precise moment by
firing a fast field pulse from a coil held over the die — no electrical
contact needed. On FaultyCat you drive it through `cat.emfi`, and unlike
crowbar glitching it also hands you back an **ADC trace** of every shot.

> ⚠️ **High voltage.** EMFI charges an HV capacitor and discharges it
> through the coil. The plastic shield is **mandatory**.

In [ ]:
import faultycat as fc
SIM = False                # True to dry-run without a board
RST_GP = None              # GP wired to target nRST; set once firmware has 'reset'
cat = fc.connect(simulator=SIM)
cat.emfi

## 1 · A single shot + ADC trace (external trigger)

Under the hood, `glitch()` runs the same three steps every time: apply your
settings, `arm()` so the HV cap charges, then `fire()`. With an external
trigger it parks in `WAITING` until it sees the target's edge. Once the shot
lands, you capture the ADC ring buffer around it.

In [ ]:
cat.emfi.trigger  = 'ext_rising'
cat.emfi.delay_us = 100
cat.emfi.width_us = 10

try:
    cat.emfi.glitch(trigger_timeout_ms=5000)
except fc.EngineError as e:
    print('engine:', e)      # e.g. HV_NOT_CHARGED / TRIGGER_TIMEOUT
print(cat.emfi.status)

In [ ]:
fc.plot_trace(cat.emfi.capture(length=512));
cat.emfi.disarm()

## 2 · Hunting the glitch — **you** program success

Just like ChipWhisperer, whether a shot actually *worked* is something you
decide by **watching the target and classifying the result** — no tool field
can tell you. So you loop over parameters, glitch, read the target back, and
tag each attempt with your own condition. The logic that defines a hit lives
in `classify()`, so edit it to match your target.

In [ ]:
def classify(resp: bytes) -> str:
    """YOUR success condition — the whole point lives here."""
    if b'root' in resp or b'OK' in resp:
        return 'success'
    if not resp:
        return 'reset'
    return 'normal'

In [ ]:
gc = fc.GlitchController(['delay', 'width'])
gc.set_range('delay', range(0, 200, 20)).set_range('width', range(1, 30, 3))

if cat.uart:
    cat.uart.open()

for p in gc.glitch_values():
    cat.emfi.trigger  = 'immediate'
    cat.emfi.delay_us = p['delay']
    cat.emfi.width_us = p['width']
    if RST_GP is not None:
        cat.target_reset(RST_GP)
    if cat.uart:
        cat.uart.reset_input()
    cat.emfi.glitch()
    resp = cat.uart.read_until(b'\n', timeout=0.1) if cat.uart else b''
    gc.add(classify(resp))

gc.counts()

In [ ]:
gc.plot(x='delay', y='width');

> **Faster, coarser alternative:** `cat.campaign('emfi')` sweeps on-device
> and streams the firmware's own `fire_status` / `verify_status` — which is
> *not* your notion of success. When you want real hits, stick with the
> observe-and-classify loop above.

In [ ]:
cat.close()